In [1]:
import geopandas as gpd

# O Miniforge já deve ter configurado o PROJ_LIB automaticamente
# Tente carregar o ficheiro usando o engine 'pyogrio'
try:
    gdf = gpd.read_file("MstCSCS_Sem_2526.gpkg", layer='ed12_polygons_all_with_heights_clean', engine='pyogrio')
    print("Sucesso! CRS detectado:", gdf.crs)
except Exception as e:
    print("Erro:", e)

Sucesso! CRS detectado: EPSG:3763


In [2]:
# mostrar as colunas de cada camada do ficheiro
import geopandas as gpd

caminho = "MstCSCS_Sem_2526.gpkg"

# 1. Listar as camadas disponíveis no ficheiro
import fiona
camadas = fiona.listlayers(caminho)
print(f"Camadas encontradas: {camadas}\n")

# 2. Ver as colunas de cada camada
for camada in camadas:
    print(f"--- Colunas da camada: {camada} ---")
    # Lemos apenas a primeira linha para ser instantâneo
    gdf_temp = gpd.read_file(caminho, layer=camada, engine='pyogrio', rows=1)
    print(gdf_temp.columns.tolist())
    print("-" * 40)

Camadas encontradas: ['ed12_polygons_all_with_heights_clean', 'avr_npolicia_pts', 'postal_code_buildings_assigned']

--- Colunas da camada: ed12_polygons_all_with_heights_clean ---
['Layer', 'PaperSpace', 'SubClasses', 'Linetype', 'EntityHandle', 'Text', 'src_file', 'texto_limpo', 'area_m2', 'source_layer', 'polygon_id', 'altura_edif_m', 'cota_topo_pt_m', 'cota_terreno_est_m', 'dist_curva_min_m', 'n_points_height', 'qa_altura_negativa', 'qa_longe_curva', 'qa_rever', 'has_height', 'geometry']
----------------------------------------
--- Colunas da camada: avr_npolicia_pts ---
['gid', 'NUMERO', 'TOPONIMIA', 'dicofre', 'DTCCFREG', 'geometry']
----------------------------------------
--- Colunas da camada: postal_code_buildings_assigned ---
['polygon_id', 'cp7', 'n_door_points', 'mean_assignment_confidence', 'mean_door_building_distance_m', 'n_cp7_on_building', 'building_area_m2', 'split_method', 'building_postal_fragment_area_m2', 'polygon_method', 'geometry']
----------------------------

In [3]:
import geopandas as gpd
from shapely.validation import make_valid

# 1. Carregar os dados
print("A carregar dados...")
caminho = "MstCSCS_Sem_2526.gpkg"
gdf = gpd.read_file(caminho, layer='ed12_polygons_all_with_heights_clean', engine='pyogrio')

# 2. Seleção de Atributos (Apenas o essencial)
# Mantemos apenas ID, altura, área e a informação de se tem altura (has_height)
colunas_uteis = ['fid', 'altura_edif_m', 'area_m2', 'has_height', 'geometry', 'cp7']
colunas_atuais = gdf.columns.tolist()
colunas_a_manter = [c for c in colunas_uteis if c in colunas_atuais]

gdf = gdf[colunas_a_manter].copy()

# 3. Correção de Geometrias
# Garante que os polígonos são válidos para SIG (corrige auto-interseções)
print("A validar e corrigir geometrias...")
gdf['geometry'] = gdf['geometry'].apply(lambda geom: make_valid(geom) if not geom.is_valid else geom)

# 4. Limpeza de registos inválidos
# Remove linhas sem geometria ou com área zero (erros de digitalização)
gdf = gdf[gdf['geometry'].notnull()]
if 'area_m2' in gdf.columns:
    gdf = gdf[gdf['area_m2'] > 0]

# 5. Guardar o ficheiro processado
output = "Edificios_Processados_Final.gpkg"
gdf.to_file(output, layer='edificios_limpos', driver="GPKG", engine='pyogrio')

print(f"\nPré-processamento concluído!")
print(f"Ficheiro guardado como: {output}")
print(f"Número total de edifícios válidos: {len(gdf)}")

A carregar dados...
A validar e corrigir geometrias...

Pré-processamento concluído!
Ficheiro guardado como: Edificios_Processados_Final.gpkg
Número total de edifícios válidos: 88649


O pré-processamento do ficheiro GeoPackage (.gpkg) dos edifícios por código postal enviado pelo Professor Paulo foi realizado para garantir a integridade dos dados e a fiabilidade das análises espaciais posteriores. 
Em primeiro lugar assegurámos que os dados estavam projetados no sistema de coordenadas EPSG:3763, que é o padrão oficial para Portugal Continental. Depois, removemos as colunas redundantes (resíduos de exportações de software CAD, como EntityHandle, Linetype e Layer), permitindo isolar apenas as variáveis de interesse para o estudo, nomeadamente a altura dos edifícios, o código-postal, a área de implantação e a geometria, resultando num conjunto de dados mais leve e eficiente para o processamento computacional.
Em adição, uma das falhas mais comuns em dados geográficos de edifícios são as geometrias inválidas (ex: polígonos que se cruzam a si mesmos). Assim, usámos o make_valid para reconstruir a topologia dos polígonos sem alterar a sua forma original. Esta correção é vital porque qualquer tentativa de realizar cruzamentos espaciais (Spatial Joins) ou operações de proximidade resultaria em erros de processamento ou na exclusão acidental de dados.
Também implementámos filtros para eliminar registos com geometria nula ou área igual a zero que podem acontecer por erros de digitalização que não representam edifícios reais.